*Módulo 5 de 9*

> **Prefer English?** Open [`05_pixels_to_parcels.ipynb`](../en/05_pixels_to_parcels.ipynb) — it is the same module, in English.


# 🧩 Módulo 5 — De píxeles a parcelas: segmentación

🧭 **Objetivos** — entender por qué clasificamos **parcelas** en vez de
píxeles sueltos, captar la intuición de **k-means** y del algoritmo de
segmentación **Shepherd**, correrlo en tu tile con `shepherd-wasm`, y ver el
campo recortado en objetos homogéneos.

📚 **¿Por qué no píxeles?** Una sola parcela son cientos de píxeles de 30 m.
Clasificar cada píxel por separado da ruido "sal y pimienta" — puntitos mal
clasificados dentro de una parcela obviamente uniforme. La clasificación
**basada en objetos** primero agrupa píxeles vecinos y espectralmente
similares en **segmentos** (parcelas), y luego clasifica cada *parcela*
completa. Mapas más limpios, y coincide con cómo funciona la agricultura de
verdad: las decisiones se toman por parcela, no por píxel.

📚 **La segmentación Shepherd** (Shepherd et al., 2019) lo hace en tres pasos:
1. **k-means** agrupa los píxeles en unas pocas "familias" espectrales.
2. **Agrupamiento (clumping)**: píxeles conexos de la misma familia se
   vuelven un segmento.
3. **Eliminación**: los segmentos menores a un umbral se fusionan con su
   vecino más parecido, para que no quede ninguna astilla suelta.

Usamos **`shepherd-wasm`**, un port en NumPy/SciPy puro que corre en el
navegador (el pipeline de escritorio usa `pyshepseg` acelerado con numba —
el mismo algoritmo, distinto motor; lo verás en el Módulo 9).

![segmentación](../../anim/es/05_segmentation.svg)


## La intuición de k-means (demo diminuta)

Antes de segmentar el tile real, siente qué hace k-means: ordena puntos en
`k` grupos por similitud. Aquí ordenamos un puñado de píxeles falsos (cada
uno con un NDVI y un índice de agua) en 3 familias espectrales. Todavía sin
geografía — solo "qué píxeles se parecen a cuáles".


In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# 9 píxeles falsos: [NDVI, índice de agua]. Tres grupos naturales.
pixeles = np.array([[0.8, 0.1], [0.82, 0.12], [0.79, 0.09],   # cultivo denso
                    [0.2, 0.1], [0.18, 0.08], [0.22, 0.11],   # suelo desnudo
                    [-0.3, 0.6], [-0.28, 0.62], [-0.31, 0.58]])# agua
familias = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(pixeles)
print("Píxel -> familia:", familias)
print("k-means halló los 3 grupos sin que le dijéramos qué son.")

In [ ]:
# Trae el tile del taller (pocos MB; queda en caché tras la primera descarga)
import os, sys

async def trae_archivo(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await trae_archivo("crop_tile_384.tif")
print("Tile listo:", TILE)

## Segmenta el tile real

`shepherd-wasm` necesita que primero importes `scipy.ndimage` y
`sklearn.cluster` (una peculiaridad de Pyodide — su autocargador no ve los
imports internos). `numClusters` fija cuántas familias espectrales siembra
k-means; `minSegmentSize` es la parcela más pequeña permitida (las menores se
fusionan). Esto tarda unos 30–60 segundos en el navegador — atento al `[*]`.


In [ ]:
# Pyodide: importa estos ANTES de shepherd_wasm para que resuelva sus internos
import scipy.ndimage, sklearn.cluster
import shepherd_wasm, time, rasterio

with rasterio.open(TILE) as src:
    img = src.read()               # (13, 384, 384)

t0 = time.time()
resultado = shepherd_wasm.doShepherdSegmentation(
    img, numClusters=30, minSegmentSize=50, imgNullVal=0, fixedKMeansInit=True)
seg = resultado.segimg.astype(np.int32)
n_seg = int(seg.max())
print(f"{n_seg} parcelas halladas en {time.time()-t0:.1f} s")

In [ ]:
import matplotlib.pyplot as plt
from scipy import ndimage

rgb = np.clip(np.dstack([img[2], img[1], img[0]]) / 3000.0, 0, 1)
# Dibuja las fronteras de parcela en amarillo sobre el color verdadero
bordes = (ndimage.maximum_filter(seg, size=2) != ndimage.minimum_filter(seg, size=2))
vis = rgb.copy(); vis[bordes] = [1, 1, 0]

plt.figure(figsize=(8, 8)); plt.imshow(vis)
plt.title(f"{n_seg} parcelas (amarillo = fronteras)"); plt.axis("off"); plt.show()
print("Cada parche con borde amarillo es un objeto que clasificaremos.")

## Experimenta

Cambia `numClusters` (prueba 15 o 50) y `minSegmentSize` (prueba 20 o 120) y
vuelve a correr las dos celdas de arriba. Menos clusters / mínimo más grande
= parcelas más grandes y gruesas; más clusters / mínimo más chico = más
detalle pero más fragmentos. No hay una única respuesta "correcta" — depende
del tamaño de las parcelas que quieras capturar.


## 🧪 Ponte a prueba

**¿Qué es el ruido "sal y pimienta", y cómo lo cura la segmentación?**

<details><summary>Ver respuesta</summary>

Son píxeles dispersos, mal clasificados de forma individual, dentro de una
parcela que en realidad es uniforme. La segmentación agrupa los píxeles de la
parcela en un objeto y clasifica el objeto completo, así unos pocos píxeles
raros no pueden motear el mapa.

</details>

**En Shepherd, ¿qué controla `minSegmentSize`, y qué pasa con las parcelas
por debajo de ese tamaño?**

<details><summary>Ver respuesta</summary>

Es el tamaño mínimo de segmento permitido. Los segmentos menores se fusionan
con su vecino espectralmente más parecido durante el paso de eliminación, así
que no sobreviven astillas diminutas.

</details>


## 🔭 Profundiza

Opcional: estas tarjetas bilingües de conceptos amplían lo que acabas
de aprender (prerrequisitos, linaje a fundamentos, referencias):

- [Segmentación de imágenes](https://abxda.github.io/rs-learning-audio/?id=image-segmentation&lang=es)
- [Segmentación (concepto)](https://abxda.github.io/rs-learning-audio/?id=segmentation&lang=es)
- [Clasificación basada en objetos](https://abxda.github.io/rs-learning-audio/?id=object-based-classification&lang=es)
- [Agrupamiento / k-means](https://abxda.github.io/rs-learning-audio/?id=clustering&lang=es)
- [Clasificación por píxel vs objeto](https://abxda.github.io/rs-learning-audio/?id=pixel-classification&lang=es)



---

[← Anterior · Módulo 4 — Índices de vegetación](04_indices_de_vegetacion.ipynb) · [Siguiente → · Módulo 6 — Las parcelas se vuelven tabla + la verdad de campo](06_variables_y_etiquetas.ipynb)
